# RÉ-EXÉCUTION v1 — YOLO26s-cls

> ⚠ **Ceci n'est PAS le notebook d'origine** (supprimé). C'est une **ré-exécution** du code `code/v1/` du dépôt, faite le jour de l'export (voir la date des sorties), avec les hyperparamètres d'origine sauvegardés (`models/.../Hyperparametres/`) et **sans nouvelle recherche d'hyperparamètres**. Les scores peuvent différer légèrement de ceux des fichiers `models/.../v1/Resultats/` (TensorFlow/CUDA non déterministes; YOLO v1 : seed 0).

Avant d'exécuter : *Add Input* → `daikhiyassine/dataset` **avant de démarrer la session**, accélérateur GPU, Internet activé. Ensuite *Save Version → Save & Run All*, puis *File → Download notebook*.

In [ ]:
# Environnement Kaggle de nov. 2026 observé : Python 3.13, 2x Tesla T4, torch 2.11.0+cu130 déjà installé -> PAS de downgrade.
# (Le downgrade torch==2.4.1+cu121 n'était nécessaire que sur GPU P100 / Python 3.12 en juillet-août 2026 ; ce paquet n'existe plus pour Python 3.13.)
!pip install -q ultralytics==8.4.110

In [ ]:
!nvidia-smi
!git clone https://github.com/Yacindaikhi/A-Cross-Architecture-Benchmark-of-DL-Models-for-Structural-Defect-Detection-on-Uncurated-Data.git repo
!git -C repo log -1 --format='%h %ad %s'

In [ ]:
# 1. Nettoyage + partition (MD5, fuites inter-splits, RGB, 256x256, JPEG q85)
!CLEAN_PATH=/kaggle/working/dataset_clean_yolo26 python repo/code/v1/data/02_clean_dataset.py

In [ ]:
# Garde-fou : arrête l'exécution si le nettoyage a échoué (évite des heures d'entraînement inutiles)
import os
root='/kaggle/working/dataset_clean_yolo26'
n={sp:sum(len(os.listdir(os.path.join(root,sp,c))) for c in os.listdir(os.path.join(root,sp))) for sp in ('train','val','test')}
print(n)
assert n=={'train': 14707, 'val': 3151, 'test': 3152}, f'effectifs inattendus : {n} (attendu {'train': 14707, 'val': 3151, 'test': 3152})'

In [ ]:
# 2. Manifeste de partition (attendu : train=14707 val=3151 test=3152)
!python repo/tools/make_manifest.py /kaggle/working/dataset_clean_yolo26 /kaggle/working/v1_clean_yolo26_manifest.csv

In [ ]:
# 3. Hyperparamètres d'origine (pas de nouvelle recherche)
!cp repo/models/yolov26_cls/v1/Hyperparametres/best_hyperparameters_yolo26.json /kaggle/working/best_hyperparameters_yolo26.json
!cat /kaggle/working/best_hyperparameters_yolo26.json

In [ ]:
# 4. Entraînement
!python repo/code/v1/yolo26/train.py

In [ ]:
# 5. Évaluation sur le split test
!python repo/code/v1/yolo26/test.py

In [ ]:
# 6. Environnement figé + empreintes des fichiers produits
!python repo/tools/capture_env.py kaggle_v1_yolo26_rerun
!python repo/tools/hash_files.py /kaggle/working/yolo26_train/aigis_yolo26_final/weights/best.pt
!cat environments/kaggle_v1_yolo26_rerun_env.json